In [ ]:
import os
import shutil
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from concurrent.futures import ThreadPoolExecutor, as_completed

# ==========================================
# 1. CONFIGURARE CAI (DOAR EyePACS)
# ==========================================
EYEPACS_DIR = Path(r"B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\originals\EyePACS")
# Citim etichetele din CSV-urile EyePACS
EYEPACS_CSVS = [EYEPACS_DIR / "original_train_labels.csv", EYEPACS_DIR / "original_test_labels.csv"]
EYEPACS_IMG_FOLDER = EYEPACS_DIR / "Images"

# Destinatia datasetului curat (doar impartit, pastrand fisierele originale)
OUTPUT_DIR = Path(r"B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_split")

CLASSES = [0, 1, 2, 3, 4]
WORKERS = max((os.cpu_count() or 2) - 1, 1)

RATIO_TRAIN = 0.70
RATIO_VAL = 0.20
RATIO_TEST = 0.10

In [ ]:
# ==========================================
# 2. FUNCTIE PENTRU COPIEREA RAPIDA
# ==========================================
def copy_file(task):
    src, dst = task
    try:
        shutil.copy2(src, dst)  # shutil.copy2 pastreaza si metadatele (data crearii etc.)
        return True, ""
    except Exception as e:
        return False, f"Eroare copiere {src.name}: {e}"

In [ ]:
# ==========================================
# 3. EXECUTIA PRINCIPALA
# ==========================================
def main():
    if OUTPUT_DIR.exists():
        print("Curatam folderul de output existent...")
        shutil.rmtree(OUTPUT_DIR)

    # Cream arhitectura de foldere (train/0, val/0, test/0 etc.)
    for split in ['train', 'val', 'test']:
        for cls in CLASSES:
            (OUTPUT_DIR / split / str(cls)).mkdir(parents=True, exist_ok=True)

    # 1. Citim si unim CSV-urile
    valid_csvs = [csv for csv in EYEPACS_CSVS if csv.exists()]
    if not valid_csvs:
        print("Eroare: Nu s-au gasit CSV-urile EyePACS la caile specificate!")
        return

    dfs = [pd.read_csv(csv) for csv in valid_csvs]
    df_all = pd.concat(dfs, ignore_index=True)
    
    # 2. Verificam care imagini exista cu adevarat pe disc
    valid_data = []
    print("\n--- 1. Verificam imaginile EyePACS pe disc ---")
    
    # Optimizare: in loc sa verificam cu .exists() care e lent, citim continutul folderului o data
    if not EYEPACS_IMG_FOLDER.exists():
        print(f"Eroare: Folderul {EYEPACS_IMG_FOLDER} nu exista.")
        return
        
    fisiere_existente = {f.name: f for f in EYEPACS_IMG_FOLDER.iterdir() if f.is_file()}
    
    for _, row in tqdm(df_all.iterrows(), total=len(df_all), desc="Scanare CSV si potrivire"):
        img_id = str(row['image'])
        label = int(row['level'])
        
        # Cautam imaginea cu extensiile comune (.jpeg este standard pentru EyePACS)
        found_path = None
        for ext in ['.jpeg', '.jpg', '.png', '.tif']:
            nume_fisier = f"{img_id}{ext}"
            if nume_fisier in fisiere_existente:
                found_path = fisiere_existente[nume_fisier]
                break
                
        if found_path:
            valid_data.append({'image_path': found_path, 'label': label, 'image_id': img_id})

    df_valid = pd.DataFrame(valid_data)
    print(f"\n✅ Total imagini EyePACS gasite si valide: {len(df_valid)}")

    if len(df_valid) == 0:
        print("Nu am gasit nicio imagine. Verifica te rog calea folderului 'Images'.")
        return

    # 3. Impartirea Stratificata 70% / 20% / 10%
    print("\n--- 2. Calculam impartirea stratificata ---")
    # Pasul A: Extragem Test-ul (10%)
    df_train_val, df_test = train_test_split(
        df_valid, 
        test_size=RATIO_TEST, 
        stratify=df_valid['label'], 
        random_state=42
    )
    
    # Pasul B: Impartim restul in Train (70% din total) si Val (20% din total)
    val_fraction = RATIO_VAL / (RATIO_TRAIN + RATIO_VAL)
    df_train, df_val = train_test_split(
        df_train_val, 
        test_size=val_fraction, 
        stratify=df_train_val['label'], 
        random_state=42
    )

    print(f"Distributie finala: TRAIN: {len(df_train)} | VAL: {len(df_val)} | TEST: {len(df_test)}")

    # 4. Generam lista de task-uri pentru copiere
    tasks = []
    
    def add_tasks(dataframe, split_name):
        for _, row in dataframe.iterrows():
            src = row['image_path']
            lbl = row['label']
            img_id = row['image_id']
            # Folosim extensia originala
            dst = OUTPUT_DIR / split_name / str(lbl) / f"{img_id}{src.suffix}"
            tasks.append((src, dst))

    add_tasks(df_train, 'train')
    add_tasks(df_val, 'val')
    add_tasks(df_test, 'test')

    # 5. Rulam multithreading-ul pentru viteza
    print(f"\n--- 3. Incepem copierea fisierelor ({len(tasks)} imagini in total) ---")
    processed, errors = 0, 0

    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        futures = [executor.submit(copy_file, task) for task in tasks]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Copiere"):
            ok, msg = future.result()
            if ok: processed += 1
            else: 
                errors += 1
                print(f"\n[Eroare] {msg}")

    print("\n" + "="*50)
    print("FINALIZAT! Toate imaginile EyePACS au fost impartite.")
    print(f"Locatia noului dataset: {OUTPUT_DIR}")

In [ ]:
if __name__ == '__main__':
    main()


--- 1. Verificam imaginile EyePACS pe disc ---


Scanare CSV si potrivire:   0%|          | 0/88702 [00:00<?, ?it/s]


✅ Total imagini EyePACS gasite si valide: 88700

--- 2. Calculam impartirea stratificata ---
Distributie finala: TRAIN: 62089 | VAL: 17741 | TEST: 8870

--- 3. Incepem copierea fisierelor (88700 imagini in total) ---


Copiere:   0%|          | 0/88700 [00:00<?, ?it/s]


FINALIZAT! Toate imaginile EyePACS au fost impartite.
Locatia noului dataset: B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_split
